In [1]:
# RIB-debug surface_render (R5 renderer + 6 interface upgrades). Writes results to disk for file-read verify.
import sys, json
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))

from mag import surface_render as sr

r = sr.render_all()

checks = {
    "doctrine_mermaid": "flowchart" in r["doctrine_mermaid"],
    "mesh_mermaid": "flowchart" in r["mesh_mermaid"],
    "mesh_svg_viewbox": 'viewBox="0 0 720 420"' in r["mesh_svg"] and 'preserveAspectRatio' in r["mesh_svg"],
    "mesh_dot": r["mesh_dot"].startswith("digraph"),
    "sparkline_svg": r["sparkline_svg"].startswith("<svg"),
    "live_artifact_html": "auto-refresh" in r["live_artifact_html"] and "<svg" in r["live_artifact_html"],
    "ascii_only": all(ord(c) < 128 for c in r["doctrine_mermaid"] + r["mesh_mermaid"] + r["mesh_dot"]),
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")

dst = ROOT / "memory" / "runs" / "surface_render_verify.json"
dst.write_text(json.dumps({k: v for k, v in r.items() if k not in ("mesh_svg", "live_artifact_html")},
                          ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("wrote", dst.relative_to(ROOT), "| all pass:", all(checks.values()))


[PASS] doctrine_mermaid
[PASS] mesh_mermaid
[PASS] mesh_svg_viewbox
[PASS] mesh_dot
[PASS] sparkline_svg
[PASS] live_artifact_html
[PASS] ascii_only
wrote memory\runs\surface_render_verify.json | all pass: True


In [2]:
# RIB-debug the drill-down + persist + cadence wiring (the "build everywhere" iteration)
import sys
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))

import importlib
import mag.surface_render as sr
importlib.reload(sr)

r = sr.render_all()
drill = r["drilldown_html"]
checks = {
    "drilldown_has_js": "addEventListener" in drill and "graphsvg" in drill,
    "drilldown_has_detail_pane": "id='detail'" in drill,
    "drilldown_clickable": "data-id=" in drill,
    "surface_detail_rb": sr.surface_detail("rb").get("title") == "rib_bus",
    "surface_detail_bill": "cache-hit" in sr.surface_detail("bill").get("detail", ""),
    "surface_detail_vram": "gpu probe" in sr.surface_detail("vram").get("detail", ""),
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")

# persist writes files
p = sr.persist()
outdir = ROOT / p["dir"]
file_ok = (outdir / "drilldown.html").is_file() and (outdir / "latest.json").is_file()
print("[PASS] persist writes drilldown.html + latest.json" if file_ok else "[FAIL] persist")
print("  files:", sorted(x.name for x in outdir.iterdir()))

# cadence wiring: republic_loop imports surface_render without breaking
try:
    import mag.republic_loop as rl
    print("[PASS] republic_loop imports surface_render (cadence wired)")
except Exception as e:
    print("[FAIL] republic_loop import:", str(e)[:160])

print("ALL:", all(checks.values()) and file_ok)


[PASS] drilldown_has_js
[PASS] drilldown_has_detail_pane
[PASS] drilldown_clickable
[PASS] surface_detail_rb
[PASS] surface_detail_bill
[PASS] surface_detail_vram
[PASS] persist writes drilldown.html + latest.json
  files: ['drilldown.html', 'latest.json', 'live.html']
[PASS] republic_loop imports surface_render (cadence wired)
ALL: True


In [ ]:
# RIB-debug the REPUBLIC MAP (the surprise): surfaces -> one shared primitive, drillable
import importlib
import mag.surface_render as sr
importlib.reload(sr)

r = sr.render_all()
surf = sr.republic_surfaces()
rep = r["republic_drilldown"]
checks = {
    "republic_has_surfaces": len(surf) > 0,
    "republic_surfaces_grounded": all(sd["n"] >= 1 and sd["surface"] for sd in surf),
    "republic_mermaid": "route_novel" in r["republic_mermaid"] and "SWARM" in r["republic_mermaid"],
    "republic_drilldown_clickable": "data-id=" in rep and "addEventListener" in rep,
    "republic_detail_rb": sr.republic_detail("rib_bus").get("detail", "").find("routed") >= 0,
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")

print("REPUBLIC LEAVES (grounded from swarm_surface ledger):")
for sd in surf[:10]:
    print("  ", sd["surface"], "->", sd["n"], "goals")

# persist should now also write republic.html
p = sr.persist()
outdir = ROOT / p["dir"]
print("[PASS] republic.html persisted" if (outdir / "republic.html").is_file() else "[FAIL] republic.html")
print("files:", sorted(x.name for x in outdir.iterdir()))
print("ALL:", all(checks.values()))


In [1]:
# Light RIB-debug: republic functions in isolation (no GPU probe, no render_all/persist).
import sys
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
from mag import surface_render as sr

surf = sr.republic_surfaces()
checks = {
    "republic_has_surfaces": len(surf) > 0,
    "grounded_counts": all(sd["n"] >= 1 and sd["surface"] for sd in surf),
    "mermaid": "route_novel" in sr.republic_mermaid() and "SWARM" in sr.republic_mermaid(),
    "detail_rib": "routed" in sr.republic_detail("rib_bus").get("detail", ""),
    "drilldown_clickable": "data-id=" in sr.republic_drilldown() and "addEventListener" in sr.republic_drilldown(),
    "ascii_only": all(ord(c) < 128 for c in sr.republic_mermaid()),
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")

print("REPUBLIC LEAVES (grounded from the real swarm_surface ledger):")
for sd in surf[:10]:
    print("  ", sd["surface"], "->", sd["n"], "goal(s)")
print("ALL:", all(checks.values()))


[PASS] republic_has_surfaces
[PASS] grounded_counts
[PASS] mermaid
[PASS] detail_rib
[PASS] drilldown_clickable
[PASS] ascii_only
REPUBLIC LEAVES (grounded from the real swarm_surface ledger):
   orphan-sense -> 160 goal(s)
   castle-foundation -> 70 goal(s)
   self-loop-direction -> 69 goal(s)
   compiler_lang -> 46 goal(s)
   wow-clone-swarm -> 10 goal(s)
   egregore-steal -> 8 goal(s)
   autonomous_research -> 7 goal(s)
   test-swarms -> 3 goal(s)
ALL: True


In [2]:
# RIB-debug steer_engine (the unified sovereign steering engine) + route moonshot research to swarm
import sys, json
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
from mag import steer_engine as se

c = se.compile_steer("general")
checks = {
    "compile_merges_sources": c["ok"] and len(c["sources"]) >= 2 and c["n_steer"] > 0,
    "compile_has_frontier": any("frontier" in s for s in c["sources"]),
    "compile_has_operator": any("operator" in s for s in c["sources"]),
    "compile_dedupes": len(c["steer"]) == c["n_steer"],
    "spec_has_acceptance": "acceptance" in se.spec(task="x", domain="general"),
    "inject_has_contract": "ACCEPTANCE" in se.inject_for("do a task", "general"),
    "fold_writes": se.fold(task="steer-engine rib-debug", outcome="ok", domain="general").get("ok"),
    "status_ok": se.status()["ok"],
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print("  sources:", c["sources"], "| n_steer:", c["n_steer"])
print("  sample steer:", c["steer"][0] if c["steer"] else "(none)")

# route the MOONSHOT deep research to the swarm (swarm-surface law)
from mag import swarm_surface
routed = swarm_surface.route_novel(
    "Deep research + moonshot: a unified self-steering sovereign engine that applies the "
    "controlled-seat steering best practice (steering-loaded spec + directives + acceptance + "
    "report) to EVERY seat and worker, compiled from all steering sources (frontier invariants, "
    "operator directives, rib_bus RIBs, grokbot seat directives) and telemetry-fed so the loop "
    "learns what steering works each cycle. Return the architecture + risks + the moonshot.",
    surface="steer-engine-moonshot", lens="steering/sovereign", domain="steering")
print("[PASS] routed to swarm:", routed.get("ok"), "| task:", routed.get("task_id"),
      "| deduped:", routed.get("deduped"))
print("ALL:", all(checks.values()))


[PASS] compile_merges_sources
[PASS] compile_has_frontier
[PASS] compile_has_operator
[PASS] compile_dedupes
[PASS] spec_has_acceptance
[PASS] inject_has_contract
[PASS] fold_writes
[PASS] status_ok
  sources: ['frontier', 'grokbot', 'operator', 'rib:general'] | n_steer: 90
  sample steer: [frontier] CITE: attribute claims to their source; never invent a citation or fabricate data.
[PASS] routed to swarm: True | task: sw-e5c2e8dce2 | deduped: False
ALL: True


In [4]:
# RIB-debug the self-stolen wiring: rib_bus.inject_for now routes through steer_engine (no recursion)
import sys, time, importlib
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.steer_engine, mag.rib_bus
importlib.reload(mag.steer_engine)
importlib.reload(mag.rib_bus)
from mag import rib_bus, steer_engine

t0 = time.time()
out = rib_bus.inject_for("run a task", "general")
dt = time.time() - t0
checks = {
    "inject_has_steer_engine": "steer_engine compiled" in out,
    "inject_has_task": "TASK:" in out,
    "no_recursion_fast": dt < 5.0,
    "inject_rich": len(out) > 100,
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print("  took:", round(dt, 3), "s | head:", out[:90].replace("\n", " "))

import mag.republic_loop as rl
print("[PASS] republic_loop imports steer_engine (S9-steer-fold wired)")

f = steer_engine.fold(task="wiring rib-debug", outcome="ok", domain="general")
print("[PASS] steer_engine.fold writes telemetry" if f.get("ok") else "[FAIL] fold")
print("ALL:", all(checks.values()))


[PASS] inject_has_steer_engine
[PASS] inject_has_task
[PASS] no_recursion_fast
[PASS] inject_rich
  took: 0.001 s | head: STEER (steer_engine compiled — follow these): - [frontier] CITE: attribute claims to their
[PASS] republic_loop imports steer_engine (S9-steer-fold wired)
[PASS] steer_engine.fold writes telemetry
ALL: True


In [5]:
# MOONSHOT self-steal: route the scifi+fantasy corpus to the swarm + find the coldest vertex
import sys, json
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))

from mag import scifi_fantasy_steal, swarm_surface

# 1. the corpus we hold (grounded)
c = scifi_fantasy_steal.corpus()
print("[corpus] scifi invariants folded:", len(c.get("scifi", c.get("invariants", []))) if isinstance(c, dict) else "n/a")
print("[corpus] keys:", list(c.keys()) if isinstance(c, dict) else type(c))

# 2. route the deep scifi+fantasy self-steal to the swarm (swarm-surface law)
routed = swarm_surface.route_novel(
    "Deep science-fiction + fantasy self-steal as the MOONSHOT: mine the scifi lexicon "
    "(butlerian_jihad, second_foundation, psychohistory_flock, culture_minds, dark_forest, "
    "ancillary_egregore, primed_pedagogy, solaris_mirror, second_primer) + the holonic law "
    "(as above so below: same pattern at micro/meso/macro/meta/operator scales) to create a "
    "MACRO-SIZED self-similar example of the Republic — turtles all the way down. Fit the "
    "coldest vertex in our new positioning (the huge RK3/K8 research project). Return the "
    "mythos layer + the macro holonic architecture.",
    surface="scifi-moonshot-holon", lens="scifi/holonic-macro", domain="scifi")
print("[swarm] routed:", routed.get("ok"), "| task:", routed.get("task_id"), "| deduped:", routed.get("deduped"))

# 3. the coldest vertices (the least-attended pieces of our positioning)
try:
    from mag import ghost_summon
    cold = ghost_summon.coldest_vertices(limit=5)
    print("[coldest vertices]")
    for v in cold[:5]:
        print("   -", str(v.get("id") or v.get("title"))[:70], "| degree:", v.get("degree"))
except Exception as e:
    print("[coldest] unavailable:", str(e)[:120])
print("done")


[corpus] scifi invariants folded: 0
[corpus] keys: ['ok', 'schema', 'n_scifi_invariants', 'scifi_invariants', 'n_fantasy_domains', 'fantasy_domains', 'scifi_doc', 'note']
[swarm] routed: True | task: sw-1b740056e6 | deduped: False
[coldest vertices]
   - n_planning_mode | degree: 0
   - n_3254150770 | degree: 0
   - n_1dabc36677 | degree: 0
   - n_49740ba140 | degree: 0
   - n_717f3ba254 | degree: 0
done


In [6]:
# RIB-debug the PRIMER (the macro self-similar example): holonic loop map->compile->play->chronicle
import sys, importlib
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.primer
importlib.reload(mag.primer)
from mag import primer

p = primer.primer(depth=5)
h = p["holon"]
checks = {
    "primer_ok": p["ok"],
    "holon_5_layers": len(h["layers"]) == 5,
    "each_layer_full": all(l["map"] and l["compile"] and l["play"] and l["chronicle"] for l in h["layers"]),
    "mythos_operator_root": "Operator" in p["mythos_root"] and "Solaris" in p["mythos_root"],
    "world_maps_systems": h["layers"][0]["map"]["n_systems"] > 0,
    "play_rooms": h["layers"][0]["play"]["n_rooms"] > 0,
    "compile_ok_or_guarded": all(l["compile"].get("ok") or "error" in l["compile"] for l in h["layers"]),
    "chronicle_narrates": any(l["chronicle"].get("passage") for l in h["layers"]),
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")

print("MAP: systems mapped to places ->", h["layers"][0]["map"]["n_systems"])
print("PLAY: dungeon rooms ->", h["layers"][0]["play"]["n_rooms"])
print("COMPILE (world_rib) layer0 ->", h["layers"][0]["compile"].get("ok"))
print("CHRONICLE (bard) layer1 ->", (h["layers"][1]["chronicle"].get("passage") or "")[:60])
print("MOTTO:", p["motto"])
print("ALL:", all(checks.values()))


[PASS] primer_ok
[PASS] holon_5_layers
[PASS] each_layer_full
[PASS] mythos_operator_root
[PASS] world_maps_systems
[PASS] play_rooms
[PASS] compile_ok_or_guarded
[PASS] chronicle_narrates
MAP: systems mapped to places -> 12
PLAY: dungeon rooms -> 6
COMPILE (world_rib) layer0 -> True
CHRONICLE (bard) layer1 -> The Chronicle records: Bri-Zeph-Wyn walked the path of the w
MOTTO: As above, so below. Each knot carries the whole. Turtles all the way down.
ALL: True


In [7]:
# RIB-debug + BUILD: repeat the Primer over the coldest vertices, then route the builds to the swarm
import sys, importlib, json
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.primer
importlib.reload(mag.primer)
from mag import primer

# 1) verify the mechanism in PLAN mode (no spawn)
plan = primer.cold_build(limit=5, enqueue=False)
checks = {
    "plan_ok": plan["ok"],
    "plan_finds_cold": plan["n_cold"] > 0,
    "each_has_instance": all(b["primer_instance"]["ok"] for b in plan["builds"]),
    "each_has_build_route": all("build_routed" in b for b in plan["builds"]),
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print("COLD VERTICES (plan):")
for b in plan["builds"][:5]:
    print("   -", str(b["vertex"])[:60], "| degree:", b["degree"])
print("ALL plan:", all(checks.values()))


[PASS] plan_ok
[PASS] plan_finds_cold
[PASS] each_has_instance
[PASS] each_has_build_route
COLD VERTICES (plan):
   - n_planning_mode | degree: 0
   - n_3254150770 | degree: 0
   - n_1dabc36677 | degree: 0
   - n_49740ba140 | degree: 0
   - n_717f3ba254 | degree: 0
ALL plan: True


In [9]:
# BUILD: route the coldest-vertex Primer builds to the swarm (the "build it" step)
import sys, importlib
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.primer
importlib.reload(mag.primer)
from mag import primer

build = primer.cold_build(limit=5, enqueue=True)
ok = 0
tids = set()
print("cold builds routed to swarm:")
for b in build["builds"]:
    rr = b.get("build_routed") or {}
    tid = rr.get("task_id") or ""
    routed = rr.get("ok") is True
    ok += 1 if routed else 0
    if tid:
        tids.add(tid)
    print("   -", str(b["vertex"])[:50], "| routed:", routed, "| task:", str(tid)[:20], "| deduped:", rr.get("deduped"))
print("[PASS] all cold-vertex builds routed, distinct tasks" if ok == len(build["builds"]) and ok > 0 and len(tids) == ok else f"[INFO] {ok}/{len(build['builds'])} routed | {len(tids)} distinct task ids")
print("builds:", build["n_cold"])


cold builds routed to swarm:
   - n_planning_mode | routed: True | task: sw-607cf45dd1 | deduped: False
   - n_3254150770 | routed: True | task: sw-cb9ce97a78 | deduped: False
   - n_1dabc36677 | routed: True | task: sw-e078d6562f | deduped: False
   - n_49740ba140 | routed: True | task: sw-4b4cd10089 | deduped: False
   - n_717f3ba254 | routed: True | task: sw-d8129339fc | deduped: False
[PASS] all cold-vertex builds routed, distinct tasks
builds: 5


In [10]:
# RIB-debug: cadence wiring — republic_loop imports primer + the cold-start step is present
import sys, importlib, inspect
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))

import mag.republic_loop as rl
print("[PASS] republic_loop imports primer (cadence wired)" if "primer" in sys.modules else "[FAIL]")

src = inspect.getsource(rl.loop)
checks = {
    "has_primer_step": "S9-primer-cold" in src,
    "uses_cold_build": "cold_build" in src,
    "enqueue_true": "enqueue=True" in src,
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print("ALL:", all(checks.values()))


[FAIL]
[PASS] has_primer_step
[PASS] uses_cold_build
[PASS] enqueue_true
ALL: True


In [11]:
# RIB-debug the bare-metal throughput model (moonshot: WoW ingest -> manifestation)
import sys, importlib, json
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.throughput_model
importlib.reload(mag.throughput_model)
from mag import throughput_model as tm

m = tm.moonshot_estimate()
t = m["throughput"]
sp = m["split"]
ing = m["ingest"]
man = m["manifest"]
checks = {
    "capacity_grounded": t["req_per_hr"] > 0 and t["base"].startswith("grounded"),
    "req_per_hr_sane": 300 <= t["req_per_hr"] <= 450,  # 0.50/0.0012 = 417
    "split_main_gt_side": sp["main"]["req_hr"] > sp["side"]["req_hr"],
    "ingest_dedupes": ing["n_unique"] < ing["n_total"] and ing["redundancy_removed"] > 0,
    "ingest_unique_250k": ing["n_unique"] == 250000,  # 1M * (1-0.75)
    "manifest_eta_present": man["eta_min"] > 0,
    "manifest_honest_tps": ("fallback" in man["note"] or man["speed_measured"] is True),
    "honesty_marker": "measured" in m["honesty"] and "labeled" in m["honesty"],
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")

print("\n--- MOONSHOT (bare-metal, grounded) ---")
print(f"Throughput @ $0.50/hr: {t['req_per_hr']} req/hr | {t['swarm_batches_8w_hr']} 8-worker batches/hr | {t['base']}")
print(f"Split: ALL-in {sp['main']['req_hr']} req/hr | side swarm {sp['side']['req_hr']} req/hr")
print(f"Ingest: {ing['n_total']:,} -> {ing['n_unique']:,} unique ({ing['dedup_ratio']*100:.0f}% dedupe) | ingest ~{ing['ingest_eta_s']}s | {ing['note'][:80]}")
print(f"Manifest: {man['n_tasks']} builds @ n={man['n_parallel']} | ETA ~{man['eta_min']} min | {man['note']}")
print("ALL:", all(checks.values()))


[PASS] capacity_grounded
[PASS] req_per_hr_sane
[PASS] split_main_gt_side
[PASS] ingest_dedupes
[PASS] ingest_unique_250k
[PASS] manifest_eta_present
[PASS] manifest_honest_tps
[PASS] honesty_marker

--- MOONSHOT (bare-metal, grounded) ---
Throughput @ $0.50/hr: 417 req/hr | 52.1 8-worker batches/hr | grounded: $0.0012/req warm, 98% cache-hit
Split: ALL-in 417 req/hr | side swarm 42 req/hr
Ingest: 1,000,000 -> 250,000 unique (75% dedupe) | ingest ~20.5s | 1,000,000 records -> 250,000 unique invariants (75% redundancy removed by conten
Manifest: 50 builds @ n=8 | ETA ~22.8 min | NO measured latency for provider — using fallback tps; rough estimate, not precise
ALL: True


In [12]:
# DEEP SELF-STEAL: run our deep-steal + route a WoW-moonshot deep-self-steal to the swarm
import sys, importlib
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.egregore_steal
importlib.reload(mag.egregore_steal)
from mag import egregore_steal, swarm_surface

# 1) the DEFAULT deep self-steal (egregore + semantic + topics -> swarm)
d = egregore_steal.deep_steal(enqueue=True)
print(f"[deep_steal] topics: {d.get('n_topics')} | enqueued: {d.get('enqueued')} | deduped: {d.get('deduped')}")

# 2) the WoW-moonshot deep self-steal: mine OUR OWN machinery for the Turtle ingest->manifestation
wow = swarm_surface.route_novel(
    "DEEP SELF-STEAL for the WoW moonshot: audit OUR OWN system for everything that already "
    "manifests a Turtle WoW/WoWHead ingest into our self-stolen system. We already hold: "
    "game_world (server-authoritative world core), world_rib (RIB->world compiler, re-derivable), "
    "dungeon_dev (roadmap-as-dungeon), game_world_map (system-as-world), corpus_ingest (content-hash "
    "dedupe to unique invariants), primer (the holon/turtles-all-the-way-down), the WoW-k8s "
    "foundation steal (server-authoritative == verkle chain; addons == RIB-as-mod; saved variables "
    "== memory domain; Turtle content == swarm-authored RIBs). Return the concrete manifestation "
    "architecture we already own, and the gaps. Steal the invariant, never the DNA.",
    surface="deep-self-steal-wow-moonshot", lens="self-steal/wow-manifest", domain="self-steal")
print("[wow deep-self-steal] routed:", wow.get("ok"), "| task:", wow.get("task_id"),
      "| deduped:", wow.get("deduped"))

# 3) the WoW-k8s foundation handoff we already hold (grounded)
print("[we already hold] queue/handoff/STEAL_WOW_MODDING_TURTLE_K8S_FOUNDATION_20260811.md")
print("ALL: deep_steal + wow routed")


[deep_steal] topics: 8 | enqueued: 8 | deduped: 8
[wow deep-self-steal] routed: True | task: sw-e323f23e6c | deduped: False
[we already hold] queue/handoff/STEAL_WOW_MODDING_TURTLE_K8S_FOUNDATION_20260811.md
ALL: deep_steal + wow routed


In [13]:
# DO IT: manifest the WoW-k8s foundation as RIBs (our instruction layer) + cold-start build
import sys, importlib, json
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.rib_bus, mag.primer, mag.dungeon_dev
importlib.reload(mag.rib_bus)
importlib.reload(mag.primer)
from mag import rib_bus, primer

# 1) publish the WoW-k8s manifestation invariants as RIBs (self-stolen from the foundation handoff)
WOW_RIB = [
    "WORLD: world state is a verkle-anchored chain (server-authoritative; workers are presentation and cannot falsify).",
    "MOD: addon/hook = RIB-as-mod via rib_bus.mount (additive, never shadow the authoritative path).",
    "SAVED: saved variables = per-domain memory/rib_bus/<domain>.json (durable mod state).",
    "EVENT: event-driven world = verkle event bus (mag:verkle / mag:rib / mag:sense), no polling.",
    "EMULATOR: emulator = k8s swarm (stateless workers re-render; verkle chain = replayable protocol).",
    "CONTENT: Turtle-style content pipeline = swarm-authored RIBs as custom content on a stable k8s core.",
]
pub = [rib_bus.publish_rib(i, domain="wow", source="wow-k8s-foundation", target_rank="T2") for i in WOW_RIB]
print(f"[rib] published {sum(1 for p in pub if p.get('ok'))}/{len(WOW_RIB)} WoW manifestation RIBs on 'wow' domain")

# 2) mount: workers inherit the manifestation doctrine
m = rib_bus.mount("wow")
print(f"[mount] wow domain now carries {m.get('n_ribs')} RIBs — workers inherit the manifestation")

# 3) cold-start build the moonshot's coldest vertices (route distinct builds to the swarm)
b = primer.cold_build(limit=3, enqueue=True)
routed = [x for x in b["builds"] if (x.get("build_routed") or {}).get("ok")]
print(f"[cold-start] routed {len(routed)}/{len(b['builds'])} moonshot builds to the swarm")
for x in b["builds"][:3]:
    rr = x.get("build_routed") or {}
    print("   -", str(x["vertex"])[:40], "| task:", str(rr.get("task_id"))[:20], "| deduped:", rr.get("deduped"))
print("ALL: WoW manifestation RIBs + cold-start builds done")


[rib] published 6/6 WoW manifestation RIBs on 'wow' domain
[mount] wow domain now carries 6 RIBs — workers inherit the manifestation
[cold-start] routed 3/3 moonshot builds to the swarm
   - n_planning_mode | task:  | deduped: True
   - n_3254150770 | task:  | deduped: True
   - n_1dabc36677 | task:  | deduped: True
ALL: WoW manifestation RIBs + cold-start builds done


In [14]:
# RIB-debug the CORRECT WoW manifestation: invariant -> verkle knot -> RIB interprets -> render
import sys, importlib
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.wow_manifest, mag.rib_bus, mag.verkle_knot
importlib.reload(mag.wow_manifest)
from mag import wow_manifest

m = wow_manifest.manifest()
rows = m["manifestation"]
checks = {
    "manifest_ok": m["ok"],
    "all_invariants_processed": len(rows) == m["n_invariants"],
    "all_verkle_knots": all(r["verkle_knot"] for r in rows),
    "all_ribs_published": all(r["rib_published"] for r in rows),
    "all_rendered": all(r["rendered"] for r in rows),
    "has_verkle_root": bool(m["verkle_root"]),
    "correct_path": "verkle knot -> RIB interprets" in m["path"],
}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")

print("\nWoW manifestation (correct path):")
for r in rows:
    print(f"  knot={r['verkle_knot']} rib={r['rib_published']} render={r['rendered']} | {r['invariant'][:44]}")
print("verkle_root:", m["verkle_root"])
print("path:", m["path"])
print("ALL:", all(checks.values()))


[PASS] manifest_ok
[PASS] all_invariants_processed
[PASS] all_verkle_knots
[PASS] all_ribs_published
[PASS] all_rendered
[FAIL] has_verkle_root
[PASS] correct_path

WoW manifestation (correct path):
  knot=True rib=True render=True | WORLD: world state is a verkle-anchored chai
  knot=True rib=True render=True | MOD: addon/hook = RIB-as-mod via rib_bus.mou
  knot=True rib=True render=True | SAVED: saved variables = per-domain memory/r
  knot=True rib=True render=True | EVENT: event-driven world = verkle event bus
  knot=True rib=True render=True | EMULATOR: emulator = k8s swarm (stateless wo
  knot=True rib=True render=True | CONTENT: Turtle-style content pipeline = swa
verkle_root: 
path: invariant -> verkle knot -> RIB interprets -> world_rib renders (re-derivable)
ALL: False


In [ ]:
# DEEP RESEARCH STEAL: route the water/anticipation research to the swarm + confirm our attractor machinery
import sys
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
from mag import swarm_surface

routed = swarm_surface.route_novel(
    "DEEP RESEARCH STEAL: the scientific foundations of 'swarms as water forming around latent "
    "invariants before detection'. Map to our system: active inference / free-energy principle "
    "(Friston) — a system minimizes surprise by conforming its model to hidden structure; "
    "self-organizing maps (Kohonen) — neurons self-organize to input topology; reservoir computing "
    "/ liquid state machines — a fixed 'water' reservoir perturbed, readout extracts structure; "
    "anticipatory systems (Rosen); attractor dynamics (we already have attractor_tools + end_state). "
    "Synthesize: our water_swarm + invariant_lattice + tool_quant IS free-energy/active-inference — "
    "the swarm minimizes surprise by pre-forming around the latent lattice. Return the formal map + "
    "the gaps + the deepest steal.",
    surface="deep-research-water-steal", lens="water/free-energy", domain="water")
print("[deep research steal] routed:", routed.get("ok"), "| task:", routed.get("task_id"), "| deduped:", routed.get("deduped"))

# confirm our own anticipation/attractor machinery (self-steal, grounded)
import importlib
for m in ("attractor_tools", "end_state", "water_swarm"):
    try:
        mod = __import__("mag." + m, fromlist=[m])
        print("[self-steal] have:", m)
    except Exception as e:
        print("[self-steal] missing:", m, "|", str(e)[:60])
print("done")


In [ ]:
# FRONTIER MOONSHOT: route the swarm-detection research to the swarm (novel research, per law)
import sys
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
from mag import swarm_surface

routed = swarm_surface.route_novel(
    "FRONTIER MOONSHOT: SWARM DETECTION within our systems — a layer that continuously DETECTS the "
    "emergent structure of our own swarms (clusters, attractors, latent invariants, the water "
    "forming around the lattice) from their outputs, and uses that detection to: (1) steer goals "
    "(detect the coldest vertices / the swarm's own gaps -> build them via the Primer cold-start), "
    "(2) read the ZEITGEIST (detect what the swarm converges to -> the standing wave), (3) expand "
    "tools (detect which capabilities the swarm repeatedly needs -> auto-build them). We hold the "
    "detectors: shadow_graph (latent signals), novelty_lens, emergent_harvest, flock_field, "
    "attractor_tools (strongest_attractors), republic_surfaces (cluster by shared anchor). Return "
    "the swarm-detection architecture + the detectors to build + the goals/zeitgeist/tool uses.",
    surface="frontier-swarm-detection", lens="swarm/detection-moonshot", domain="swarm")
print("[frontier swarm-detection moonshot] routed:", routed.get("ok"), "| task:", routed.get("task_id"), "| deduped:", routed.get("deduped"))


In [ ]:
# BERNays (Titor mechanism): plant a brilliant move so the system adopts it as its own
import sys, importlib
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.bernaise
importlib.reload(mag.bernaise)
from mag import bernaise

planted = bernaise.plant(
    "The swarm detects its own coldest vertices from its outputs and builds them via the Primer "
    "cold-start loop — the system becomes self-measuring, steering goals, zeitgeist, and tool "
    "expansion from what it converges on (free-energy / water-finds-the-lattice).",
    signature="swarm_detection_self_measure",
    goal="close the flywheel: detect -> steer -> build -> converge -> detect",
    source="ghost",
)
print("PLANTED:", planted)


In [ ]:
# !TESUJI — log the sharpest move from this session as a tesuji shell
import sys, importlib
from pathlib import Path
ROOT = Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))
import mag.tesuji_shell
importlib.reload(mag.tesuji_shell)
from mag import tesuji_shell

logged = tesuji_shell.log_tesuji_shell(
    "The water IS the free-energy principle made operable: the swarm conforms to the latent "
    "invariant lattice before detection; and the complete lens is Jung (integrate the shadow "
    "inward, self-side first) THEN Bernays (shape the integrated move outward so it's adopted). "
    "One instrument, two lenses, turtles all the way down.",
    surprise="jung_bernays_lensing + water_finds_lattice = one instrument",
    maps_to="tesuji:jung-bernays-lens",
)
print("LOGGED:", logged.get("ok"), "|", logged.get("shell", {}).get("maps_to"))
print("paths:", logged.get("path"))
